# 03_Hybrid_Model_Training

## Purpose

This notebook trains the proposed Hybrid ViT–Swin classifier using the extracted feature tensors generated in `02_Feature_Extraction.ipynb`. It loads the saved features, constructs the hybrid classifier, trains with early stopping, and saves the best-performing model.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import precision_recall_fscore_support

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

CLASS_NAMES = [
    "Adbhuta", "Bhayanaka", "Bibhatsya",
    "Hasya", "Karuna", "Raudra",
    "Shanta", "Shringara", "Veera"
]
NUM_CLASSES = len(CLASS_NAMES)


In [ ]:
BASE_DIR = "/content/drive/MyDrive/NAVARASA_PROJECT_DRIVE1"

FEATURE_DIR = os.path.join(BASE_DIR, "extracted_features")
MODEL_DIR = os.path.join(BASE_DIR, "models", "hybrid")
os.makedirs(MODEL_DIR, exist_ok=True)


In [ ]:
train_data = torch.load(os.path.join(FEATURE_DIR, "train.pt"))
val_data   = torch.load(os.path.join(FEATURE_DIR, "val.pt"))
test_data  = torch.load(os.path.join(FEATURE_DIR, "test.pt"))

X_swin_tr, X_vit_tr, y_tr = train_data.values()
X_swin_vl, X_vit_vl, y_vl = val_data.values()
X_swin_te, X_vit_te, y_te = test_data.values()


In [ ]:
train_ds = TensorDataset(X_swin_tr, X_vit_tr, y_tr)
val_ds   = TensorDataset(X_swin_vl, X_vit_vl, y_vl)
test_ds  = TensorDataset(X_swin_te, X_vit_te, y_te)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=64)
test_dl  = DataLoader(test_ds, batch_size=64)


In [ ]:
class HybridClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(1024 + 768, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, NUM_CLASSES)
        )

    def forward(self, s, v):
        return self.fc(torch.cat([s, v], dim=1))


In [ ]:
model = HybridClassifier().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_f1 = 0
wait = 0
patience = 5


In [ ]:
for epoch in range(30):
    model.train()
    loss_sum = 0

    for s, v, y in train_dl:
        s, v, y = s.to(device), v.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(s, v)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item()

    model.eval()
    preds, gt = [], []

    with torch.no_grad():
        for s, v, y in val_dl:
            pred = model(s.to(device), v.to(device)).argmax(1)
            preds.extend(pred.cpu())
            gt.extend(y)

    _, _, f1, _ = precision_recall_fscore_support(
        gt, preds, average="macro", zero_division=0
    )

    print(f"Epoch {epoch+1:02d} | Loss {loss_sum/len(train_dl):.4f} | Val F1 {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        wait = 0
        save_path = os.path.join(MODEL_DIR, "best_hybrid.pt")
        torch.save(model.state_dict(), save_path)
        print(f"Best model saved to: {save_path}")
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping")
            break

print(f"Best Validation Macro F1: {best_f1:.4f}")
